# GPT-J-6B IKE / GRACE Reproduction - One-Click Google Colab Notebook (v1)

Reproduce the two **memory-based baselines** of CALM (CISC7021 Group26 W2, Tang I Choi) on GPT-J-6B (fp16) using EasyEdit:

- **IKE** (In-Context Knowledge Editing, Zheng et al. 2023): non-parametric. Retrieves k=16 in-context examples from a fact memory bank (zsre_mend_train) with a sentence-transformer and prepends them to the prompt at generation time. No weights are changed.
- **GRACE** (Gradient Ascent with Contrastive Regularization, Hartvigsen et al. 2023): performs n_iter=200 gradient-ascent edits with early-stop contrastive regularization on a single MLP weight.

Both are driven through the same `BaseEditor` interface and scored under the same unified protocol (rewrite / rephrase / locality) as the teammate ROME reproduction, so numbers are directly comparable.

**How to run**: Runtime -> Run all. Idempotent: existing clone / model / data are skipped on re-run.
**Tuning**: set ALG / MODEL / DATA / N_CASES / TRAIN_SIZE in cell 3.1.


In [ ]:
# 1.1 Clone the project repository (robust: retry git clone, fallback to zip download)
import os, subprocess, sys, time, zipfile, urllib.request

REPO_URL = "https://github.com/FongChinWai-Vivi/CALM-Editor-CISC7021-Task1-Group26.git"
ZIP_URL = "https://github.com/FongChinWai-Vivi/CALM-Editor-CISC7021-Task1-Group26/archive/refs/heads/main.zip"
PROJ_DIR = "/content/CALM-Editor-CISC7021-Task1-Group26"
ZIP_PATH = "/content/repo.zip"
EXTRACT_DIR = "/content"
EXTRACTED_NAME = "CALM-Editor-CISC7021-Task1-Group26-main"

def clone_with_retry():
    for attempt in range(1, 4):
        try:
            print(f"Cloning project repository (attempt {attempt}/3) ...")
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, PROJ_DIR], check=True)
            return True
        except subprocess.CalledProcessError as e:
            print(f"git clone failed (attempt {attempt}/3): {e}")
            if attempt < 3:
                time.sleep(5)
    return False

def clone_via_zip():
    print("git clone failed after 3 attempts, falling back to zip download ...")
    urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    extracted = os.path.join(EXTRACT_DIR, EXTRACTED_NAME)
    if os.path.exists(PROJ_DIR):
        print("Target directory already exists, keeping it.")
    else:
        os.rename(extracted, PROJ_DIR)
    print("Repository extracted from zip archive.")

if not os.path.isdir(PROJ_DIR):
    if not clone_with_retry():
        clone_via_zip()
else:
    print("Project directory already exists, skipping clone.")
print("Project directory:", PROJ_DIR)


In [ ]:
# 1.2 Check that the W2 IKE/GRACE reproduction script and hparams are in place
script_path = os.path.join(PROJ_DIR, "EasyEdit", "examples", "w2_ike_grace_repro.py")
ike_hp = os.path.join(PROJ_DIR, "EasyEdit", "hparams", "IKE", "gpt-j-6B.yaml")
grace_hp = os.path.join(PROJ_DIR, "EasyEdit", "hparams", "GRACE", "gpt-j-6b.yaml")

missing = [p for p in (script_path, ike_hp, grace_hp) if not os.path.isfile(p)]
if not missing:
    print("w2_ike_grace_repro.py, IKE/gpt-j-6B.yaml and GRACE/gpt-j-6b.yaml all exist, ready to run.")
else:
    print("The repository is missing the following files, please upload them:")
    for p in missing:
        print("  -", p)
    from google.colab import files
    up = files.upload()
    import shutil
    for name in up.keys():
        if name.endswith("w2_ike_grace_repro.py"):
            os.makedirs(os.path.dirname(script_path), exist_ok=True)
            shutil.move(name, script_path)
        elif name.endswith("gpt-j-6B.yaml") and "IKE" in name or name == "gpt-j-6B.yaml":
            os.makedirs(os.path.dirname(ike_hp), exist_ok=True)
            shutil.move(name, ike_hp)
        elif name.endswith("gpt-j-6b.yaml"):
            os.makedirs(os.path.dirname(grace_hp), exist_ok=True)
            shutil.move(name, grace_hp)
    print("Upload complete.")


In [ ]:
# 1.3 Check torch version (skip reinstall by default, keep CUDA check)
import torch

print("Current torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

SKIP_TORCH_REINSTALL = True
MIN_TORCH = "2.1.0"

def version_tuple(v):
    nums = []
    for x in v.split(".")[:3]:
        try:
            nums.append(int(x))
        except ValueError:
            break
    while len(nums) < 3:
        nums.append(0)
    return tuple(nums)

cur = torch.__version__.split("+")[0]
if version_tuple(cur) >= version_tuple(MIN_TORCH):
    print(f"torch {cur} is compatible, skipping reinstall.")
else:
    print(f"torch {cur} is older than {MIN_TORCH}; Colab default torch is fine, no action needed.")


In [ ]:
# 1.4 Install ALL EasyEdit dependencies (one-shot, fixes the import chain)
# EasyEdit's easyeditor/__init__.py -> dataset -> trainer.utils -> blip2_models
# pulls in BLIP2 deps: iopath, fairscale, av, qwen_vl_utils, hydra-core, ...
import importlib, os, re, subprocess, sys

req_path = os.path.join(PROJ_DIR, "EasyEdit", "requirements.txt")
filtered_req = os.path.join(PROJ_DIR, "EasyEdit", "requirements_no_torch.txt")
with open(req_path, "r", encoding="utf-8") as f:
    lines = f.readlines()
kept = [ln for ln in lines if not re.match(r"^\s*(torch|torchvision|torchaudio)([=@<>\s]|$)", ln, re.IGNORECASE)]
with open(filtered_req, "w", encoding="utf-8") as f:
    f.writelines(kept)

print("Installing requirements (torch-related lines excluded) ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", filtered_req], check=False)

# Extra packages needed by the BLIP2 import chain (not in requirements.txt)
EXTRA = ["iopath", "fairscale", "av", "qwen_vl_utils", "hydra-core", "timm", "accelerate", "peft", "higher"]  # higher is required by MEND.py (import chain)
print("Installing extra deps:", EXTRA)
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + EXTRA, check=False)

# Verify the import chain works; auto-fill anything still missing (max 5 rounds)
os.chdir(os.path.join(PROJ_DIR, "EasyEdit"))
sys.path.insert(0, os.getcwd())
PKG_ALIAS = {"hydra": "hydra-core"}  # easyeditor imports hydra; the pip package is hydra-core

for attempt in range(5):
    try:
        import easyeditor
        print("easyeditor import OK - all dependencies satisfied.")
        break
    except ModuleNotFoundError as e:
        name = str(e).split("'")[1].split(".")[0]
        install = PKG_ALIAS.get(name, name)
        print(f"missing {name} -> pip install {install}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", install], check=False)
else:
    print("Still missing packages after 5 rounds.")
    raise SystemExit(1)


In [ ]:
# 2.1 Check disk space
!df -h /content


In [ ]:
# 2.2 Download the model (idempotent; keeps *.bin and detects pytorch_model.bin)
from huggingface_hub import snapshot_download

MODEL = "gpt-j-6B"   # gpt-j-6B (fp16 mirror, ~11.2GB) / gpt2-xl (~3GB)
if MODEL == "gpt-j-6B":
    HF_REPO = "ainize/gpt-j-6B-float16"
    model_dir = os.path.join(PROJ_DIR, "EasyEdit", "hugging_cache", "gpt-j-6B")
else:
    HF_REPO = "gpt2-xl"
    model_dir = os.path.join(PROJ_DIR, "EasyEdit", "hugging_cache", "gpt2-xl")

def has_model():
    if not os.path.isdir(model_dir):
        return False
    if any(f.endswith(".safetensors") for f in os.listdir(model_dir)):
        return True
    if os.path.isfile(os.path.join(model_dir, "pytorch_model.bin")):
        return True
    return os.path.isfile(os.path.join(model_dir, "pytorch_model.bin.index.json"))

if has_model():
    print("Model already exists, skipping download.")
else:
    print(f"Downloading {HF_REPO} (~11.2GB for GPT-J-6B fp16, 5-20 min depending on bandwidth)...")
    os.makedirs(model_dir, exist_ok=True)
    snapshot_download(
        repo_id=HF_REPO,
        local_dir=model_dir,
        allow_patterns=["*.json", "*.txt", "*.safetensors", "*.bin", "*.md", "*.gitattributes"],
    )
    print("Model download complete.")
    for f in sorted(os.listdir(model_dir))[:20]:
        print(" -", f)


In [ ]:
import os
import sys
# Ensure the data directory exists before the script downloads datasets
os.makedirs(os.path.join(PROJ_DIR, "EasyEdit", "data"), exist_ok=True)

# 3.0 Sync the reproduction script to the latest remote version (Colab clone may be stale)
import subprocess
subprocess.run(["git", "-C", PROJ_DIR, "checkout", "--", "EasyEdit/examples/w2_ike_grace_repro.py"], check=False)
subprocess.run(["git", "-C", PROJ_DIR, "pull"], check=False)

# 3.1 Run IKE or GRACE (full output captured to /content/w2_ike_grace_run.log)
import os
import torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

ALG        = "ike"     # ike / grace
MODEL      = "gpt-j-6B"  # gpt-j-6B / gpt2-xl  (must match cell 2.2 MODEL)
DATA       = "zsre"    # counterfact / zsre
N_CASES    = 10        # 10 for smoke test, 50 for full run
TRAIN_SIZE = 10000     # IKE memory bank size (top-N of zsre_mend_train; 0 = full train set; GRACE ignores it)
USE_FP16   = True      # keep True on T4 for GPT-J-6B

cmd = [
    sys.executable, "examples/w2_ike_grace_repro.py",
    "--alg", ALG,
    "--model", MODEL,
    "--data", DATA,
    "--n", str(N_CASES),
    "--device", "cuda:0",
]
if ALG == "ike":
    cmd += ["--train-size", str(TRAIN_SIZE)]
if USE_FP16:
    cmd.append("--fp16")

print("Executing command:", " ".join(cmd))
print("Note: first run includes data download + IKE memory-bank encoding; a single GRACE edit runs 200 gradient steps, keep the tab open.")
log_path = "/content/w2_ike_grace_run.log"
with open(log_path, "w") as logf:
    result = subprocess.run(cmd, cwd=os.path.join(PROJ_DIR, "EasyEdit"),
                            stdout=logf, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
print(f"{ALG.upper()} run exit code:", result.returncode)
print("Full log saved:", log_path)
with open(log_path) as f:
    lines = f.readlines()
print("total log lines:", len(lines))
for ln in lines[-40:]:
    print(ln, end="")


In [ ]:
# 4.1 Copy the latest result to /content and print a metric summary (matches script fields)
import glob, json, shutil

res_dir = os.path.join(PROJ_DIR, "EasyEdit", "results", "eval")
pattern = f"{ALG.upper()}_{MODEL}_*.json"
jsons = sorted(glob.glob(os.path.join(res_dir, pattern)), key=os.path.getmtime)
if not jsons:
    raise FileNotFoundError(f"No result JSON matching {pattern}. Check the previous step (3.1) log.")

latest = jsons[-1]
dst = f"/content/{ALG.upper()}_{MODEL}_latest.json"
shutil.copy(latest, dst)
print("Latest result file:", os.path.basename(latest))
print("Copied to:", dst)

with open(latest, "r", encoding="utf-8") as f:
    data = json.load(f)

print("\n===== Metric Summary =====")
print("alg          :", data.get("alg"))
print("model        :", data.get("model"))
print("split        :", data.get("split"))
print("rewrite_acc  :", data.get("rewrite_acc"))
print("rephrase_acc :", data.get("rephrase_acc"))
print("locality     :", data.get("locality"))
print("portability  :", data.get("portability"))
print("memory_bank  :", data.get("memory_bank"))
print("fp16         :", data.get("fp16"))
print("elapsed_s    :", data.get("elapsed_s"))
print("peak_vram_mb :", data.get("peak_vram_mb"))


In [ ]:
# 4.2 Option 1: Save to Google Drive (optional - does NOT block Run All)
# Drive mounting pops an authorization dialog. To enable Drive saving, set
# SAVE_TO_DRIVE=true in this cell (or an earlier one), otherwise Run All skips it.
import os
if os.environ.get("SAVE_TO_DRIVE", "").lower() in ("1", "true", "yes"):
    from google.colab import drive
    drive.mount("/content/drive")
    import shutil
    drive_dir = f"/content/drive/MyDrive/CISC7021_{ALG.upper()}"
    os.makedirs(drive_dir, exist_ok=True)
    shutil.copy(dst, os.path.join(drive_dir, os.path.basename(dst)))
    print("Saved result JSON to Drive:", drive_dir)
else:
    print("Drive save skipped (set SAVE_TO_DRIVE=true to enable).")


In [ ]:
# 4.3 Option 2: Download directly to the local browser
from google.colab import files
files.download(dst)
print("Download triggered; if nothing happens, manually download from the left file panel: " + dst)


## Notes

- **v1**: one-click Run All notebook for the two W2 memory-based baselines (IKE + GRACE), mirroring the teammate ROME notebook structure.
- IKE does **not** modify weights: it prepends k=16 retrieved facts to the prompt (memory bank = top-N of zsre_mend_train). `--train-size 10000` matches the EasyEdit IKE example; the truncated slice is cached under `EasyEdit/data/`.
- GRACE performs 200 gradient-ascent steps with early-stop contrastive regularization on `transformer.h[25].mlp.fc_out.weight` (GPT-J-6B) or `transformer.h[35].mlp.c_fc.weight` (gpt2-xl).
- Locality slice: CounterFact uses `neighborhood_prompts` (neighborhood key); zsRE uses the official `loc` slice (nq key). Both match `docs/evaluation_protocol.md`.
- Expected runtime on Colab T4 16GB: IKE n=10 ≈ a few minutes (encoding 10k facts is the slow part, one-time); GRACE n=10 ≈ 15-40 min (200 iters per case).
- Results are saved under `EasyEdit/results/eval/{IKE|GRACE}_{model}_{timestamp}.json`; `results/` is git-ignored, keep a copy for the report.
